# CacheMind — Governed Semantic Cache Training on Google Colab (GPU)

**Team Members:**
1. Dhanya Shree M (2403717672622012)
2. Deva Sri R (2403717672622010)

### Objectives (from Machine Learning Project Abstract):
1. **Dense Semantic Representation:** Pre-trained Sentence Transformer (`all-MiniLM-L6-v2`, 384-D, Cosine similarity).
2. **Fast Similarity Search:** FAISS (Facebook AI Similarity Search, `IndexFlatIP`).
3. **Semantic Equivalence Classification:** Logistic Regression using 8 domain features.
4. **Post-Hoc Probability Calibration:** Isotonic Regression calibrating raw classifier scores into true posterior match probabilities.
5. **Risk-Based Threshold Selection:** Clopper-Pearson exact binomial confidence bound targeting False-Hit Rate (FHR) $\le 1\%$.
6. **Unsupervised Query Clustering:** K-Means clustering for frequent query topic discovery.

---
### Benchmark Datasets:
- **Public Benchmark:** Quora Question Pairs (QQP) [Hugging Face Link](https://huggingface.co/datasets/nyu-mll/glue/viewer/qqp)
- **Enterprise Domain Dataset:** Tamil Nadu Enterprise Policy Pairs (`enterprise_policy_pairs_split.csv`)

In [ ]:
# Step 1: Install Dependencies on Colab
!pip install -q sentence-transformers faiss-cpu scikit-learn scipy pandas numpy joblib datasets

In [ ]:
# Step 2: Verify GPU Acceleration
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active device on Colab: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 3: Load Quora Question Pairs (QQP) Benchmark Dataset
from datasets import load_dataset
import pandas as pd

print('Loading QQP benchmark dataset from Hugging Face...')
qqp_dataset = load_dataset('glue', 'qqp')
print(qqp_dataset)

train_df = pd.DataFrame(qqp_dataset['train'][:10000]) # 10k sample for fast GPU training
val_df = pd.DataFrame(qqp_dataset['validation'][:2000])
print(f'Train shape: {train_df.shape}, Validation shape: {val_df.shape}')

In [ ]:
# Step 4: Encode Sentence Embeddings using all-MiniLM-L6-v2
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

def extract_features(df):
    q1_texts = df['question1'].tolist()
    q2_texts = df['question2'].tolist()
    
    embs1 = model.encode(q1_texts, normalize_embeddings=True, show_progress_bar=True)
    embs2 = model.encode(q2_texts, normalize_embeddings=True, show_progress_bar=True)
    
    # Cosine similarity
    cos_sims = np.sum(embs1 * embs2, axis=1)
    
    # Jaccard lexical overlap
    jaccards = []
    for q1, q2 in zip(q1_texts, q2_texts):
        t1, t2 = set(str(q1).lower().split()), set(str(q2).lower().split())
        j = len(t1 & t2) / len(t1 | t2) if (t1 | t2) else 0.0
        jaccards.append(j)
        
    X = np.column_stack([cos_sims, np.array(jaccards)])
    y = np.array(df['label'])
    return X, y

X_train, y_train = extract_features(train_df)
X_val, y_val = extract_features(val_df)

In [ ]:
# Step 5: Train Logistic Regression Equivalence Classifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, max_iter=500))
])

pipeline.fit(X_train, y_train)
raw_val_scores = pipeline.predict_proba(X_val)[:, 1]
print('Logistic Regression classifier trained successfully!')

In [ ]:
# Step 6: Fit Isotonic Regression Probability Calibrator
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip')
calibrator.fit(raw_val_scores[:1000], y_val[:1000])

cal_probs = calibrator.predict(raw_val_scores[1000:])
print('Brier Score Before:', brier_score_loss(y_val[1000:], raw_val_scores[1000:]))
print('Brier Score After:', brier_score_loss(y_val[1000:], cal_probs))

In [ ]:
# Step 7: Clopper-Pearson Exact Binomial Upper Confidence Bound
from scipy.stats import beta

def clopper_pearson_ucb(k, n, confidence=0.95):
    if n == 0 or k == n:
        return 1.0
    alpha = 1.0 - confidence
    return float(beta.ppf(1.0 - alpha, k + 1, n - k))

target_fhr = 0.01 # 1% False-Hit Rate Target
print('Evaluating candidate acceptance thresholds...')
for tau in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    hits = (cal_probs >= tau)
    n_hits = np.sum(hits)
    if n_hits > 0:
        false_hits = np.sum(y_val[1000:][hits] == 0)
        obs_fhr = false_hits / n_hits
        ucb = clopper_pearson_ucb(false_hits, n_hits)
        print(f'Tau: {tau:.2f} | Hits: {n_hits} | False Hits: {false_hits} | Obs FHR: {obs_fhr:.3f} | 95% UCB: {ucb:.3f}')

In [ ]:
# Step 8: Save Trained Artifacts to Download and Put into CacheMind
import joblib

joblib.dump(pipeline, 'classifier_pipeline.joblib')
joblib.dump(calibrator, 'isotonic_calibrator.joblib')

print('Saved classifier_pipeline.joblib and isotonic_calibrator.joblib')
print('Download these files from Colab and place in backend/artifacts/models/')